# 1 Imports

In [ ]:
import joblib
import pandas as pd

import funciones

# Audio de entrada

In [ ]:
audio = 'ruta/al/audio.wav'

## Extraer características

In [ ]:
caracteristicas = funciones.extract_features(audio)
df = pd.DataFrame([caracteristicas])

## Extraer gráficas

# Cargar modelos

In [ ]:
modelo_rf = joblib.load('Jorge/modelo_Random_Forest.pkl')
modelo_lgbm = joblib.load('Jorge/modelo_LightGBM.pkl')
modelo_svm = joblib.load('Jorge/modelo_SVM.pkl')

## Cargar datos de entrenamiento

In [ ]:
X_train = pd.read_csv('datos_entrenamiento.csv')

# Random Forest

## Generar predicciones

In [ ]:
y_pred_rf = modelo_rf.predict(df)
y_pred_proba_rf = modelo_rf.predict_proba(df)[:, 1]

## Generar explicaciones

### SHAP
Hay que ver bien cómo hacer esta función para que funcione ya que usa X_train y X_test (de donde los sacamos)

In [ ]:
funciones.explica_shap(modelo=modelo_rf, )

# LightGBM

## Generar predicciones

In [ ]:
y_pred_lgbm = modelo_lgbm.predict(df)
y_pred_proba_lgbm = modelo_lgbm.predict_proba(df)[:, 1]

# Support Vector Machine

## Generar predicciones

In [ ]:
y_pred_svm = modelo_svm.predict(df)
y_pred_proba_svm = modelo_svm.predict_proba(df)[:, 1]

# WAV2VEC2

Cosas que necesita el modelo

In [ ]:
import torch
from datasets import load_dataset, Audio, Value
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
import numpy as np
import librosa

In [10]:
base = os.getcwd()

In [ ]:
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    os.path.join(base,"Iván\wav2vec2-deepfake-final") # Poner ruta donde esté el modelo
)
model.eval()

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    os.path.join(base,"Iván//wav2vec2-deepfake-final") # Poner ruta donde esté el modelo
)

In [ ]:
SAMPLING_RATE = 16000
MAX_AUDIO_LEN = 10 * SAMPLING_RATE

def preprocess_single(audio_path, max_audio_len, sampling_rate, feature_extractor):
    # 1. Cargar audio
    audio, _ = librosa.load(audio_path, sr=sampling_rate)

    # 2. Recorte (igual que en tu dataset)
    if len(audio) > max_audio_len:
        audio = audio[:max_audio_len]

    # 3. Feature extraction
    inputs = feature_extractor(
        audio,
        sampling_rate=sampling_rate,
        padding="max_length",
        max_length=max_audio_len,
        truncation=True,
        return_tensors="pt"
    )

    return inputs["input_values"]  # tensor listo para el modelo

In [ ]:
input_values = preprocess_single(
    audio,
    MAX_AUDIO_LEN,
    SAMPLING_RATE,
    feature_extractor
)

# Inferencia
with torch.no_grad():
    outputs = model(input_values)

logits = outputs.logits

print(logits)

tensor([[ 0.3211, -0.3416]])


In [15]:
def logits_to_score(logits):
    x1, x2 = logits[0]
    return 1 / (1 + np.exp(-(x2 - x1)))

In [ ]:
print("nombre =", os.path.splitext(os.path.basename(audio))[0])
print("logits =", logits)
print("score =", float(logits_to_score(logits))
)

nombre = I_32_N
logits = tensor([[ 0.3211, -0.3416]])
score = 0.3401312232017517
